In [1]:
!mamba install pandas

mambajs 0.21.1

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 2.0203000000000464 seconds
  Name           Version  Build                Channel
--------------------------------------------------------------------
+ pandas         3.0.3    np23py313h1e705a5_0  emscripten-forge-4x
+ python-tzdata  2026.2   pyhd8ed1ab_0         conda-forge
- pip            26.1.2   pyh145f28c_0         conda-forge


In [2]:
import pandas as pd

In [3]:
orders = pd.read_csv("orders.csv")
pricing = pd.read_csv("product_pricing.csv")
catalog = pd.read_csv("product_catalog.csv")

In [4]:
df = orders.merge(
    pricing,
    on=["product_name", "channel"],
    how="left"
)

In [7]:
df = df.merge(
    catalog[["product_name", "category"]],
    on="product_name",
    how="left"
)

In [17]:
df["revenue"] = (
    df["MRP"]
    * df["quantity"]
    * (1 - df["discount_pct"] / 100)
)

df = df.rename(columns={"category_x": "category"})

In [18]:
result = (
    df.groupby("category")
      .agg(
          Total_Revenue=("revenue", "sum"),
          Orders=("order_id", "count")
      )
      .reset_index()
)

result["Avg_Order_Value"] = (
    result["Total_Revenue"] /
    result["Orders"]
)

result = result[
    [
        "category",
        "Total_Revenue",
        "Orders",
        "Avg_Order_Value"
    ]
]

In [19]:
print(result)

   category  Total_Revenue  Orders  Avg_Order_Value
0  Haircare       38013.82      14      2715.272857
1    Makeup       15283.39       5      3056.678000
2  Skincare       64722.59      15      4314.839333
3  Wellness       65337.06      16      4083.566250


In [21]:
# KPI Metrics
total_revenue = round(df["revenue"].sum(), 2)
total_orders = df["order_id"].nunique()
avg_order_value = round(total_revenue / total_orders, 2)
total_customers = df["customer_id"].nunique()

# Revenue by Category
category_df = (
    df.groupby("category")["revenue"]
      .sum()
      .reset_index()
)

# Revenue by Channel
channel_df = (
    df.groupby("channel")["revenue"]
      .sum()
      .reset_index()
)

# Top Products
product_df = (
    df.groupby("product_name")
      .agg(
          revenue=("revenue","sum"),
          orders=("order_id","count")
      )
      .reset_index()
      .sort_values("revenue", ascending=False)
      .head(10)
)

In [43]:
import json

category_json = category_df.to_json(
    orient="records"
)

channel_json = channel_df.to_json(
    orient="records"
)

products_json = product_df.to_json(
    orient="records"
)

   category   revenue
0  Haircare  38013.82
1    Makeup  15283.39
2  Skincare  64722.59
3  Wellness  65337.06
     channel   revenue
0     Amazon  14004.96
1        App  78385.95
2  Instagram  44409.55
3    Website  25920.24
4   WhatsApp  20636.16
           product_name   revenue  orders
16      Vitamin C Serum  30730.35       6
1   Collagen Supplement  20429.18       5
2           Conditioner  14298.75       3
0           Ashwagandha  14100.00       3
10          Night Cream  13540.80       3
Rows in df: 54
Rows in category_df: 4
Rows in channel_df: 5
Rows in product_df: 10


In [47]:
# ==========================
# HTML DASHBOARD
# ==========================

html = f"""
<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<title>D2C Dashboard</title>

<script src="https://cdn.jsdelivr.net/npm/chart.js"></script>

<style>

body {{
    font-family: Arial, sans-serif;
    background: #f4f6f9;
    padding: 20px;
}}

h1 {{
    margin-bottom: 20px;
}}

.kpis {{
    display: grid;
    grid-template-columns: repeat(4,1fr);
    gap: 15px;
    margin-bottom: 30px;
}}

.card {{
    background: white;
    padding: 20px;
    border-radius: 10px;
    text-align: center;
}}

.card h3 {{
    color: gray;
}}

.card p {{
    font-size: 24px;
    font-weight: bold;
}}

.charts {{
    display: grid;
    grid-template-columns: 2fr 1fr;
    gap: 20px;
}}

.chart-card {{
    background: white;
    padding: 20px;
    border-radius: 10px;
}}

table {{
    width: 100%;
    border-collapse: collapse;
}}

th, td {{
    padding: 10px;
    border-bottom: 1px solid #ddd;
}}

th {{
    background: #222;
    color: white;
}}

.chart-container{{
    position: relative;
    height: 400px;
    width: 100%;
}}

canvas{{
    max-height: 400px;
}}
</style>

</head>

<body>

<h1>D2C Brand Dashboard</h1>

<div class="kpis">

<div class="card">
<h3>Total Revenue</h3>
<p>₹{total_revenue:,.2f}</p>
</div>

<div class="card">
<h3>Total Orders</h3>
<p>{total_orders:,}</p>
</div>

<div class="card">
<h3>Average Order Value</h3>
<p>₹{avg_order_value:,.2f}</p>
</div>

<div class="card">
<h3>Total Customers</h3>
<p>{total_customers:,}</p>
</div>

</div>

<div class="charts">

<div class="chart-card">
<h3>Revenue by Category</h3>
<div class="chart-container">
    <canvas id="categoryChart"></canvas>
</div>
</div>

<div class="chart-card">
<h3>Revenue by Channel</h3>
<div class="chart-container">
    <canvas id="channelChart"></canvas>
</div>
</div>

</div>

<br><br>

<div class="chart-card">

<h3>Top Products</h3>

<table>

<thead>
<tr>
<th>Product</th>
<th>Revenue</th>
<th>Orders</th>
</tr>
</thead>

<tbody id="productTable"></tbody>

</table>

</div>

<script>

const categoryData = {category_json};

const channelData = {channel_json};

const productData = {products_json};

// Category Chart


new Chart(
    document.getElementById('categoryChart'),
    {{
        type: 'bar',
        data: {{
            labels: categoryData.map(x => x.category),
            datasets: [{{
                label: 'Revenue',
                data: categoryData.map(x => x.revenue)
            }}]
        }},
        options: {{
            responsive: true,
            maintainAspectRatio: false
        }}
    }}
);


// Channel Chart

new Chart(
document.getElementById('channelChart'),
{{
type:'pie',
data:{{
labels: channelData.map(x => x.channel),
datasets:[{{
data: channelData.map(x => x.revenue)
}}]
}}
}}
);

// Product Table

let rows = "";

console.log(productData);
productData.forEach(p => {{
rows += `
<tr>
<td>${{p.product_name}}</td>
<td>₹${{Number(p.revenue).toLocaleString()}}</td>
<td>${{p.orders}}</td>
</tr>
`;
}});

document.getElementById("productTable").innerHTML = rows;

</script>

</body>
</html>
"""

In [48]:
with open(
    "Dashboard.html",
    "w",
    encoding="utf-8"
) as f:
    f.write(html)